# Browser agent (multi-turn) — Browserless × LangChain (Python)

**Tier 2.** Multi-turn `browserless_agent` workflow over streamable HTTP: navigate → snapshot → click → re-snapshot → extract. Each step is a separate `tools/call` request, so they must share the same MCP session and land on the same droplet.

## Important: Python session pattern

The Python `langchain-mcp-adapters` library opens a **fresh** MCP `ClientSession` for every `tool.ainvoke()` call by default — see the [adapter README](https://github.com/langchain-ai/langchain-mcp-adapters#streamable-http). That works fine for stateless tools (smartscraper, search, etc.) but breaks stateful agent loops, because each call gets a different `Mcp-Session-Id` and the load balancer routes it to a different browser.

To preserve state, wrap the multi-turn calls in a `client.session(...)` context manager and bind tools to that session via `load_mcp_tools(session)`. That's what this notebook does.

(The JS adapter `@langchain/mcp-adapters` reuses the connection by default — see `js/browser_agent.ts` for the simpler shape.)

In [ ]:
%pip install -q langchain-mcp-adapters langgraph langchain-anthropic

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic

MCP_URL = os.environ.get("BROWSERLESS_MCP_URL", "https://mcp.browserless.io/mcp")

client = MultiServerMCPClient({
    "browserless": {
        "transport": "http",
        "url": MCP_URL,
        "headers": {"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    }
})

## Direct multi-turn proof

Three sequential `browserless_agent` calls inside a single MCP session: `goto` → `snapshot` → `text`. Call 3 must return `"Example Domain"` — proof that the navigation from call 1 persisted into call 3.

In [ ]:
async with client.session("browserless") as session:
    tools = await load_mcp_tools(session)
    agent_tool = next(t for t in tools if t.name == "browserless_agent")

    await agent_tool.ainvoke({"method": "goto", "params": {"url": "https://example.com"}})
    await agent_tool.ainvoke({"method": "snapshot"})
    result = await agent_tool.ainvoke({"method": "text", "params": {"selector": "h1"}})
    print(result)
    assert "Example Domain" in str(result), "Session state was not preserved between calls"

## Full ReAct loop

Hand all 10 tools to a LangGraph ReAct agent. The model decides when to use the multi-turn `browserless_agent` vs. a single-shot stateless tool. The agent loop runs inside the same `client.session(...)` block so every tool call shares one MCP session.

In [ ]:
async with client.session("browserless") as session:
    tools = await load_mcp_tools(session)

    agent = create_react_agent(
        ChatAnthropic(model="claude-sonnet-4-6"),
        tools,
    )

    prompt = (
        "Use browserless_agent to navigate to https://news.ycombinator.com, click the first story link, "
        "and report the title and first paragraph of that page. Use snapshot/click/text methods step by step."
    )

    out = await agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
    print(out["messages"][-1].content)